# 001 OSM Network Feasibility Validation Plan

神奈川県の道路ネットワーク一括取得に向けた技術検証ノートブック。

- 対象都道府県: 神奈川県
- 想定業務: ラストマイル配送
- 取得対象ネットワーク: `drive`
- 対応する計画書: `docs/brainstorming/001_osm_network_feasibility_validation_plan.md`


## 出力方針

- OSM から取得した生に近い道路ネットワーク: `data/raw/road_network/`
- 後処理済みネットワーク: `data/processed/`
- 検証メトリクス: `outputs/metrics/`
- 可視化結果: `outputs/figures/`

このノートブックでは、まず `data/raw/road_network/` に神奈川県一括取得結果を保存する前提で進める。


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'road_network'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'

for path in [DATA_RAW_DIR, DATA_PROCESSED_DIR, OUTPUT_METRICS_DIR, OUTPUT_FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_RAW_DIR, OUTPUT_METRICS_DIR, OUTPUT_FIGURES_DIR


(WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/data/raw/road_network'),
 WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/outputs/metrics'),
 WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/outputs/figures'))

In [7]:
PREFECTURE_NAME = 'Kanagawa, Japan'
NETWORK_TYPE = 'drive'
GRAPH_FILENAME = 'kanagawa_drive.graphml'
METRICS_FILENAME = 'kanagawa_drive_network_metrics.csv'

GRAPH_PATH = DATA_RAW_DIR / GRAPH_FILENAME
METRICS_PATH = OUTPUT_METRICS_DIR / METRICS_FILENAME

print(f'Prefecture: {PREFECTURE_NAME}')
print(f'Network type: {NETWORK_TYPE}')
print(f'Graph path: {GRAPH_PATH}')
print(f'Metrics path: {METRICS_PATH}')


Prefecture: Kanagawa, Japan
Network type: drive
Graph path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\road_network\kanagawa_drive.graphml
Metrics path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_drive_network_metrics.csv


## 次の実装セルで行うこと

1. `osmnx` で神奈川県の `drive` ネットワークを取得する
2. ノード数、エッジ数、処理時間、ファイルサイズを記録する
3. `GraphML` で保存し、再読込できることを確認する


In [5]:
import time

import osmnx as ox
import pandas as pd


In [ ]:
start_time = time.perf_counter()

graph = ox.graph_from_place(PREFECTURE_NAME, network_type=NETWORK_TYPE)

elapsed_seconds = time.perf_counter() - start_time

ox.save_graphml(graph, GRAPH_PATH)
reloaded_graph = ox.load_graphml(GRAPH_PATH)

file_size_mb = GRAPH_PATH.stat().st_size / (1024 * 1024)
metrics_df = pd.DataFrame(
    [
        {
            'prefecture_name': PREFECTURE_NAME,
            'network_type': NETWORK_TYPE,
            'node_count': graph.number_of_nodes(),
            'edge_count': graph.number_of_edges(),
            'elapsed_seconds': round(elapsed_seconds, 2),
            'graph_path': str(GRAPH_PATH),
            'file_size_mb': round(file_size_mb, 2),
            'reload_node_count': reloaded_graph.number_of_nodes(),
            'reload_edge_count': reloaded_graph.number_of_edges(),
        }
    ]
)

metrics_df.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
metrics_df


In [ ]:
print(f'Download completed: {GRAPH_PATH}')
print(f'Metrics saved: {METRICS_PATH}')
print(f'Graph file size (MB): {file_size_mb:.2f}')
print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')
print(f'Reload nodes: {reloaded_graph.number_of_nodes()}')
print(f'Reload edges: {reloaded_graph.number_of_edges()}')
print(f'Elapsed seconds: {elapsed_seconds:.2f}')


Download completed: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\road_network\kanagawa_drive.graphml
Metrics saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_drive_network_metrics.csv
Graph file size (MB): 259.52
Nodes: 223850
Edges: 608205
Reload nodes: 223850
Reload edges: 608205
Elapsed seconds: 296.38


## Step 3. 県全体ネットワークの後処理

このセルでは、Step 2 で取得した生の `GraphML` を、後続の距離計算や VRP 前処理に使いやすい形へ整える。

実施内容:
- 生ネットワークから最大弱連結成分のみを残す
- 車配送の routing に必要な属性を残し、それ以外を削減する
- `maxspeed` をもとに `travel_time` を付与する
- 後処理済み `GraphML` を `data/processed/kanagawa_drive_step3/` に保存する
- 保存後に再読込し、ノード・エッジ・属性差分を確認する
- 実行ログを `logs/` に保存し、メトリクスを表示する


In [8]:
from datetime import datetime
from pathlib import Path
import math

import networkx as nx

STEP3_OUTPUT_DIR = DATA_PROCESSED_DIR / 'kanagawa_drive_step3'
STEP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

processed_graph_path = STEP3_OUTPUT_DIR / 'kanagawa_drive_processed.graphml'
processed_metrics_path = STEP3_OUTPUT_DIR / 'kanagawa_drive_processed_metrics.csv'
timestamp_label = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = LOG_DIR / f'001_osm_network_feasibility_validation_plan_step3_{timestamp_label}.log'

keep_graph_attrs = {'created_date', 'created_with', 'crs', 'simplified'}
keep_node_attrs = {'x', 'y', 'street_count', 'highway', 'junction', 'ref'}
keep_edge_attrs = {
    'osmid',
    'highway',
    'oneway',
    'reversed',
    'length',
    'access',
    'lanes',
    'maxspeed',
    'name',
    'ref',
    'geometry',
    'bridge',
    'tunnel',
    'junction',
    'travel_time',
}


def normalize_value(value):
    if hasattr(value, 'wkt'):
        return value.wkt
    if isinstance(value, float):
        if math.isnan(value):
            return 'NaN'
        return round(value, 6)
    if isinstance(value, (list, tuple, set)):
        return tuple(normalize_value(item) for item in value)
    if isinstance(value, dict):
        return tuple(sorted((key, normalize_value(val)) for key, val in value.items()))
    return value


def prune_graph_attributes(graph):
    graph.graph = {key: value for key, value in graph.graph.items() if key in keep_graph_attrs}
    for _, data in graph.nodes(data=True):
        remove_keys = [key for key in data if key not in keep_node_attrs]
        for key in remove_keys:
            del data[key]
    for _, _, _, data in graph.edges(keys=True, data=True):
        remove_keys = [key for key in data if key not in keep_edge_attrs]
        for key in remove_keys:
            del data[key]


def compare_graphs(left_graph, right_graph, max_differences=20):
    differences = []

    left_graph_attrs = {key: normalize_value(value) for key, value in left_graph.graph.items()}
    right_graph_attrs = {key: normalize_value(value) for key, value in right_graph.graph.items()}
    if left_graph_attrs != right_graph_attrs:
        differences.append(
            f'Graph attributes differ: left={left_graph_attrs}, right={right_graph_attrs}'
        )

    left_nodes = set(left_graph.nodes)
    right_nodes = set(right_graph.nodes)
    only_left_nodes = sorted(left_nodes - right_nodes)
    only_right_nodes = sorted(right_nodes - left_nodes)
    for node_id in only_left_nodes[:max_differences]:
        differences.append(f'Node only in processed graph: {node_id}')
    for node_id in only_right_nodes[:max_differences]:
        differences.append(f'Node only in reloaded graph: {node_id}')

    shared_nodes = sorted(left_nodes & right_nodes)
    for node_id in shared_nodes:
        left_attrs = {key: normalize_value(value) for key, value in left_graph.nodes[node_id].items()}
        right_attrs = {key: normalize_value(value) for key, value in right_graph.nodes[node_id].items()}
        if left_attrs != right_attrs:
            differences.append(
                f'Node attribute difference at {node_id}: left={left_attrs}, right={right_attrs}'
            )
        if len(differences) >= max_differences:
            return differences[:max_differences]

    left_edges = set(left_graph.edges(keys=True))
    right_edges = set(right_graph.edges(keys=True))
    only_left_edges = sorted(left_edges - right_edges)
    only_right_edges = sorted(right_edges - left_edges)
    for edge_key in only_left_edges[:max_differences]:
        differences.append(f'Edge only in processed graph: {edge_key}')
    for edge_key in only_right_edges[:max_differences]:
        differences.append(f'Edge only in reloaded graph: {edge_key}')

    shared_edges = sorted(left_edges & right_edges)
    for edge_key in shared_edges:
        u, v, k = edge_key
        left_attrs = {
            key: normalize_value(value) for key, value in left_graph.get_edge_data(u, v, k).items()
        }
        right_attrs = {
            key: normalize_value(value) for key, value in right_graph.get_edge_data(u, v, k).items()
        }
        if left_attrs != right_attrs:
            differences.append(
                f'Edge attribute difference at {(u, v, k)}: left={left_attrs}, right={right_attrs}'
            )
        if len(differences) >= max_differences:
            return differences[:max_differences]

    return differences[:max_differences]


step3_start_time = time.perf_counter()
raw_graph = ox.load_graphml(GRAPH_PATH)
component_count_before = nx.number_weakly_connected_components(raw_graph)
largest_component_nodes = max(nx.weakly_connected_components(raw_graph), key=len)
processed_graph = raw_graph.subgraph(largest_component_nodes).copy()
removed_nodes = raw_graph.number_of_nodes() - processed_graph.number_of_nodes()
removed_edges = raw_graph.number_of_edges() - processed_graph.number_of_edges()

processed_graph = ox.add_edge_speeds(processed_graph)
processed_graph = ox.add_edge_travel_times(processed_graph)
prune_graph_attributes(processed_graph)

ox.save_graphml(processed_graph, processed_graph_path)
reloaded_processed_graph = ox.load_graphml(processed_graph_path)
step3_elapsed_seconds = time.perf_counter() - step3_start_time
processed_file_size_mb = processed_graph_path.stat().st_size / (1024 * 1024)
component_count_after = nx.number_weakly_connected_components(processed_graph)

differences = compare_graphs(processed_graph, reloaded_processed_graph)

metrics_step3_df = pd.DataFrame(
    [
        {
            'input_graph_path': str(GRAPH_PATH),
            'processed_graph_path': str(processed_graph_path),
            'raw_node_count': raw_graph.number_of_nodes(),
            'raw_edge_count': raw_graph.number_of_edges(),
            'processed_node_count': processed_graph.number_of_nodes(),
            'processed_edge_count': processed_graph.number_of_edges(),
            'removed_nodes': removed_nodes,
            'removed_edges': removed_edges,
            'component_count_before': component_count_before,
            'component_count_after': component_count_after,
            'processed_file_size_mb': round(processed_file_size_mb, 2),
            'elapsed_seconds': round(step3_elapsed_seconds, 2),
            'difference_count': len(differences),
        }
    ]
)
metrics_step3_df.to_csv(processed_metrics_path, index=False, encoding='utf-8-sig')

retained_node_attributes = sorted({key for _, data in processed_graph.nodes(data=True) for key in data.keys()})
retained_edge_attributes = sorted(
    {key for _, _, _, data in processed_graph.edges(keys=True, data=True) for key in data.keys()}
)

log_lines = [
    'Step 3 post-processing log',
    f'Created at: {datetime.now().isoformat()}',
    f'Input graph: {GRAPH_PATH}',
    f'Processed graph: {processed_graph_path}',
    f'Processed metrics: {processed_metrics_path}',
    f'Raw nodes: {raw_graph.number_of_nodes()}',
    f'Raw edges: {raw_graph.number_of_edges()}',
    f'Processed nodes: {processed_graph.number_of_nodes()}',
    f'Processed edges: {processed_graph.number_of_edges()}',
    f'Removed nodes: {removed_nodes}',
    f'Removed edges: {removed_edges}',
    f'Weakly connected components before: {component_count_before}',
    f'Weakly connected components after: {component_count_after}',
    f'Elapsed seconds: {step3_elapsed_seconds:.2f}',
    f'Processed file size (MB): {processed_file_size_mb:.2f}',
    f'Retained node attributes: {retained_node_attributes}',
    f'Retained edge attributes: {retained_edge_attributes}',
]

if differences:
    log_lines.append('Differences found between processed graph and reloaded graph:')
    log_lines.extend(differences)
else:
    log_lines.append('No differences found between processed graph and reloaded graph.')

log_path.write_text('\n'.join(log_lines) + '\n', encoding='utf-8')

print(f'Step 3 output directory: {STEP3_OUTPUT_DIR}')
print(f'Processed graph saved: {processed_graph_path}')
print(f'Processed metrics saved: {processed_metrics_path}')
print(f'Log saved: {log_path}')
print(f'Raw nodes: {raw_graph.number_of_nodes()}')
print(f'Raw edges: {raw_graph.number_of_edges()}')
print(f'Processed nodes: {processed_graph.number_of_nodes()}')
print(f'Processed edges: {processed_graph.number_of_edges()}')
print(f'Removed nodes: {removed_nodes}')
print(f'Removed edges: {removed_edges}')
print(f'Weakly connected components before: {component_count_before}')
print(f'Weakly connected components after: {component_count_after}')
print(f'Processed file size (MB): {processed_file_size_mb:.2f}')
print(f'Elapsed seconds: {step3_elapsed_seconds:.2f}')
if differences:
    print(f'Differences found after reload: {len(differences)}')
    for difference in differences:
        print(difference)
else:
    print('Differences found after reload: none')

metrics_step3_df


Step 3 output directory: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3
Processed graph saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3\kanagawa_drive_processed.graphml
Processed metrics saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3\kanagawa_drive_processed_metrics.csv
Log saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step3_20260317_170057.log
Raw nodes: 223850
Raw edges: 608205
Processed nodes: 223850
Processed edges: 608205
Removed nodes: 0
Removed edges: 0
Weakly connected components before: 1
Weakly connected components after: 1
Processed file size (MB): 285.75
Elapsed seconds: 160.85
Differences found after reload: none


,input_graph_path,processed_graph_path,raw_node_count,raw_edge_count,processed_node_count,processed_edge_count,removed_nodes,removed_edges,component_count_before,component_count_after,processed_file_size_mb,elapsed_seconds,difference_count
0,C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,223850,608205,223850,608205,0,0,1,1,285.75,160.85,0
